# 0. import

In [18]:
import sys
import os, time, torch, tqdm, gc, numpy as np, pandas as pd

import torch.nn as nn
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from torch.amp import autocast, GradScaler         
from torch.nn.utils import clip_grad_norm_
import torch.nn.functional as F

from torch_geometric.utils import negative_sampling
from torch.optim.lr_scheduler import ReduceLROnPlateau

from neo4j.exceptions import ServiceUnavailable
from neomodel import config as neomodel_config
from neomodel import db

from app.config.local import NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD # src/app/config/local.py
NEO4J_USER     = NEO4J_USER
NEO4J_PWD      = DB_PASSWORD
NEO4J_HOST     = NEO4J_HOST
NEO4J_PORT     = NEO4J_PORT

neomodel_config.DATABASE_URL = f'bolt://{NEO4J_USER}:{NEO4J_PWD}@{NEO4J_HOST}:{NEO4J_PORT}'

try:
    result, _ = db.cypher_query("RETURN 1 AS test")
    print("✅ Neo4j 연결 성공! → 응답값:", result[0][0])
except ServiceUnavailable as e:
    print("❌ Neo4j 연결 실패:", e)

✅ Neo4j 연결 성공! → 응답값: 1


### 기초 함수 정의

In [19]:
# 단계별 경과 시간을 콘솔에 출력.
def log(step: str, start: float):
    dur = time.perf_counter() - start
    print(f"[{step}] ✅ 완료  |  {dur:6.2f} s")
START_ALL = time.perf_counter()

# convert cypher to df(pandas)
def cypher_df(q: str, params: dict | None = None) -> pd.DataFrame:
    records, cols = db.cypher_query(q, params or {})
    df = pd.DataFrame(records, columns=cols)

    # 1) 어떤 컬럼을 변환할지 결정
    id_cols = [c for c in df.columns
               if c.endswith('_id') or c in {'mid', 'oid', 'pid'}]

    # 2) 캐스팅 (문자·공백·NULL → NaN → Int64 → int64)
    for col in id_cols:
        df[col] = (
            pd.to_numeric(df[col], errors="coerce")  # 문자열·공백 처리
              .astype("Int64")                      # Nullable 정수
              .astype("int64")                      # 확정 int64
        )
    return df

# enable safe mapping(drop rows when failure)
def safe_map(series: pd.Series, mapping: dict[int,int], col: str) -> np.ndarray:
    mapped = series.map(mapping)
    miss   = mapped.isna().sum()
    if miss:
        print(f"⚠️  {col}: {miss:,}개 매핑 실패 → 행 제거")
        mapped = mapped.dropna()
    return mapped.astype(np.int64).to_numpy()



In [ ]:
records, _ = db.cypher_query("""
MATCH (m:Member)
RETURN m.member_id AS mid
ORDER BY mid
""")
mids = [mid for (mid,) in records]        

# mids를 int형으로 변환 후 numpy array로
mids_arr = np.array([int(mid) for mid in mids])

min_id, max_id = mids_arr.min(), mids_arr.max()
missing = set(range(min_id, max_id + 1)) - set(mids_arr)
is_serialized = len(missing) == 0 and (np.diff(np.sort(mids_arr)) == 1).all()

print("min_id:", min_id)
print("max_id:", max_id)
print("missing:", sorted(missing))
print("직렬화 여부:", is_serialized)


In [ ]:
# Neo4j에서 order_id 목록 가져오기
records, _ = db.cypher_query("MATCH (o:Order) RETURN o.order_id")
order_ids = [int(oid) for (oid,) in records]

# numpy로 변환 후 직렬화 여부 검사
import numpy as np
order_arr = np.array(order_ids)
min_oid, max_oid = order_arr.min(), order_arr.max()
missing_order = set(range(min_oid, max_oid + 1)) - set(order_arr)
is_serialized_order = len(missing_order) == 0 and (np.diff(np.sort(order_arr)) == 1).all()

print("🧾 [Order] ID 범위:", min_oid, "~", max_oid)
print("⛔ 빠진 Order ID:", sorted(missing_order))
print("✅ 직렬화 여부 (Order):", is_serialized_order)

### 이거 없는 이유 : eval_set = test인 주문을 삭제

In [ ]:
records, _ = db.cypher_query("MATCH (p:Product) RETURN p.product_id")
product_ids = [int(pid) for (pid,) in records]

product_arr = np.array(product_ids)
min_pid, max_pid = product_arr.min(), product_arr.max()
missing_product = set(range(min_pid, max_pid + 1)) - set(product_arr)
is_serialized_product = len(missing_product) == 0 and (np.diff(np.sort(product_arr)) == 1).all()

print("🛒 [Product] ID 범위:", min_pid, "~", max_pid)
print("⛔ 빠진 Product ID:", sorted(missing_product))
print("✅ 직렬화 여부 (Product):", is_serialized_product)

### 이거 없는 이유 : 애초에 주문 내역이 없어서 병합 과정에서 삭제됨

## 데이터 불러오기

### 데이터 직렬화(인덱스 정렬)

In [3]:
t0 = time.perf_counter()

# 1) 노드-ID 추출 ────────────────
df_member  = cypher_df("MATCH (m:Member)  RETURN m.member_id  AS mid")
df_order   = cypher_df("MATCH (o:Order)   RETURN o.order_id   AS oid")
df_product = cypher_df("MATCH (p:Product) RETURN p.product_id AS pid")

# 2) dict ↔ index 매핑 ──────────
m2i = {mid: i for i, mid in enumerate(df_member.mid.unique())}
o2i = {oid: i for i, oid in enumerate(df_order.oid.unique())}
p2i = {pid: i for i, pid in enumerate(df_product.pid.unique())}

# 역매핑: NumPy 배열로  O(1) 룩업
i2m = np.fromiter(m2i.keys(), dtype=np.int64)
i2o = np.fromiter(o2i.keys(), dtype=np.int64)
i2p = np.fromiter(p2i.keys(), dtype=np.int64)

print(f"users={len(m2i):,} | orders={len(o2i):,} | products={len(p2i):,}")
log("ID→Index 매핑", t0)

users=206,209 | orders=3,346,083 | products=49,685
[ID→Index 매핑] ✅ 완료  |  230.02 s


### 피쳐 로드

In [4]:
t0 = time.perf_counter()

# 2-1. Order feature  ────────────────────────────
df_order_feat = cypher_df("""
MATCH (o:Order)
RETURN o.order_id AS oid,
       o.days_since_prior_order_norm AS dsp,
       o.order_dow_norm              AS dow,
       o.order_hour_of_day_norm      AS hr,
       o.order_number_norm           AS idx
""")

# ※ 매핑 실패 행 제거 (oid → o2i)
df_order_feat["oid_idx"] = safe_map(df_order_feat.oid, o2i, "oid")
order_feat = (
    df_order_feat
      .sort_values("oid_idx")[["dsp","dow","hr","idx"]]
      .to_numpy(np.float32)
)                              

# 2-2. Product feature  ──────────────────────────
df_prod_vec = cypher_df("""
MATCH (p:Product)
RETURN p.product_id AS pid,
       p.category_vector + p.name_vector AS vec
""")

df_prod_vec["pid_idx"] = safe_map(df_prod_vec.pid, p2i, "pid")
prod_feat = torch.tensor(
    np.stack(df_prod_vec.sort_values("pid_idx").vec.values),
    dtype=torch.float32
)                                             # shape = [N_prod, D]
prod_feat_dim = prod_feat.size(1)

print(f"Order feature  : {order_feat.shape}")
print(f"Product vector : {prod_feat.shape}")
log("피처 로딩", t0)


Order feature  : (3346083, 4)
Product vector : torch.Size([49685, 2048])
[피처 로딩] ✅ 완료  |  292.73 s


### 학습용 서브그래프

In [5]:
t0 = time.perf_counter()

# prior (구조적 간선: Member→Order→Product)
df_edge_tr = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.role_train = 'prior'
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")
uo = torch.stack((
    torch.from_numpy(safe_map(df_edge_tr.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge_tr.oid, o2i, "oid"))
), 0).to(torch.long)
op = torch.stack((
    torch.from_numpy(safe_map(df_edge_tr.oid, o2i, "oid")),
    torch.from_numpy(safe_map(df_edge_tr.pid, p2i, "pid"))
), 0).to(torch.long)
print(f"Train-prior edges : {uo.size(1):,}")

# Positive label (user-buys-product, supervised 학습 라벨)
df_label_tr = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.role_train = 'train'
RETURN m.member_id AS mid, p.product_id AS pid
""")
pos_u = torch.from_numpy(safe_map(df_label_tr.mid, m2i, "mid"))
pos_p = torch.from_numpy(safe_map(df_label_tr.pid, p2i, "pid"))
edge_pos = torch.stack((pos_u, pos_p), 0).to(torch.long)
print(f"Positive pairs    : {edge_pos.size(1):,}")

# Negative sampling
from torch_geometric.utils import negative_sampling
NEG_K = 10
num_neg = edge_pos.size(1) * NEG_K
edge_neg = negative_sampling(
    edge_index=edge_pos,
    num_nodes=(len(m2i), len(p2i)),
    num_neg_samples=num_neg
).to(torch.long)
print(f"Negative samples  : {edge_neg.size(1):,}")

# Label edge_index/edge_label (user-buys-product)
edge_label_index = torch.cat([edge_pos, edge_neg], 1)
edge_label = torch.cat([
    torch.ones(edge_pos.size(1)),
    torch.zeros(edge_neg.size(1))
])

log("서브그래프 구성", t0)

# ───────────────────────────────────────────────
# 2. HeteroData 객체 완성 (학습용)
# ───────────────────────────────────────────────
data = HeteroData()
data['user'].x    = nn.Embedding(len(m2i), 32).weight
data['order'].x   = order_feat
data['product'].x = prod_feat
data['user'].num_nodes, data['order'].num_nodes, data['product'].num_nodes = \
    len(m2i), len(o2i), len(p2i)

# 구조적 간선 (2-hop)
data['user','ordered','order'].edge_index        = uo
data['order','ordered_by','user'].edge_index     = uo.flip(0)
data['order','contains','product'].edge_index    = op
data['product','contained_in','order'].edge_index= op.flip(0)

# 라벨용 직접간선 (user-buys-product, 1-hop)
data['user','buys','product'].edge_index         = edge_label_index

Train-prior edges : 30,883,423
Positive pairs    : 770,266
Negative samples  : 7,702,660
[서브그래프 구성] ✅ 완료  |  725.21 s


In [6]:
# ── HeteroData ─────────────────────────────────────────────────────────
data = HeteroData()
data['user'].x    = nn.Embedding(len(m2i), 32).weight          # 32-dim 임베딩
data['order'].x   = order_feat                                 # 4-dim 실수
data['product'].x = prod_feat                                  # D-dim 실수
data['user'].num_nodes, data['order'].num_nodes, data['product'].num_nodes = \
    len(m2i), len(o2i), len(p2i)

data['user','ordered','order'].edge_index       = uo
data['order','ordered_by','user'].edge_index    = uo.flip(0)
data['order','contains','product'].edge_index   = op
data['product','contained_in','order'].edge_index = op.flip(0)

### 테스트용 서브그래프 

In [7]:
# ────────────────────────────────────────────────────────────────
# 0. 테스트용 prior/label 간선 및 gt_dict 생성 (negative X)
# ────────────────────────────────────────────────────────────────

# 1) prior 간선 (eval_set='test', role_test='prior')
df_edge_te = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='prior'
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")
uo_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid"))
), 0).to(torch.long)
op_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid")),
    torch.from_numpy(safe_map(df_edge_te.pid, p2i, "pid"))
), 0).to(torch.long)

# 3) 테스트용 HeteroData
test_data = HeteroData()
test_data['user'].num_nodes    = len(m2i)
test_data['order'].num_nodes   = len(o2i)
test_data['product'].num_nodes = len(p2i)
test_data['user'].x    = nn.Embedding(len(m2i), 32).weight
test_data['order'].x   = torch.tensor(order_feat, dtype=torch.float32)
test_data['product'].x = prod_feat
test_data['user','ordered','order'].edge_index         = uo_te
test_data['order','ordered_by','user'].edge_index      = uo_te.flip(0)
test_data['order','contains','product'].edge_index     = op_te
test_data['product','contained_in','order'].edge_index = op_te.flip(0)

### df_gt = validation data

In [8]:
# 1. ground-truth: 각 test user의 정답 product 인덱스 리스트
df_gt = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='train'
RETURN m.member_id AS mid, collect(p.product_id) AS gt_list
""")

mid_arr = safe_map(df_gt.mid, m2i, "mid")
# 각 gt_list가 파이썬 리스트(list) 형태 → 각 리스트별로 개별적으로 매핑
gt_list_arr = [
    [p2i[pid] for pid in pid_list if pid in p2i]
    for pid_list in df_gt.gt_list
]
gt_dict = {
    int(mid): [int(pid) for pid in pid_list]
    for mid, pid_list in zip(mid_arr, gt_list_arr) if mid != -1
}
print(f"GT users (index): {len(gt_dict):,}")


GT users (index): 75,000


# 2. 모델 정의

In [9]:
# HetSAGE 모델 (dropout 적용)
class HetSAGE(nn.Module):
    def __init__(self, hid=128, out=128, p_drop=0.2, aggr='sum'):
        super().__init__()
        self.conv1 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
        }, aggr=aggr)
        self.conv2 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((hid, hid), out, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
        }, aggr=aggr)
        self.norm1 = nn.LayerNorm(hid)
        self.norm2 = nn.LayerNorm(out)
        self.p_drop = p_drop

    def _drop(self, h: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        return {k: F.dropout(v, p=self.p_drop, training=self.training) for k, v in h.items()}

    def forward(self, x_dict, edge_index_dict):
        h = self.conv1(x_dict, edge_index_dict)
        h = {k: F.relu(self.norm1(v)) for k, v in h.items()}
        h = self._drop(h)
        h = self.conv2(h, edge_index_dict)
        h = {k: F.relu(self.norm2(v)) for k, v in h.items()}
        h = self._drop(h)
        return h

In [10]:
"""
# HetSAGE 모델 정의
class HetSAGE(nn.Module):
    def __init__(self, hid: int = 64, out: int = 64):
        super().__init__()
        # ① 1-hop
        self.conv1 = HeteroConv({
            ('user',   'ordered',      'order'):   SAGEConv((-1, -1), hid),
            ('order',  'contains',     'product'): SAGEConv((-1, -1), hid),
            ('order',  'ordered_by',   'user'):    SAGEConv((-1, -1), hid),  # 역
            ('product','contained_in', 'order'):   SAGEConv((-1, -1), hid)   # 역
        }, aggr='mean')
        # ② 2-hop
        self.conv2 = HeteroConv({
            ('user',   'ordered',      'order'):   SAGEConv((hid, hid), out),
            ('order',  'contains',     'product'): SAGEConv((hid, hid), out),
            ('order',  'ordered_by',   'user'):    SAGEConv((hid, hid), out),
            ('product','contained_in', 'order'):   SAGEConv((hid, hid), out)
        }, aggr='mean')

    def forward(self, x_dict, edge_dict):
        h = {k: torch.relu(v) for k, v in self.conv1(x_dict, edge_dict).items()}
        return self.conv2(h, edge_dict)                 # h_dict ⟶ 최종 임베딩
        """

"\n# HetSAGE 모델 정의\nclass HetSAGE(nn.Module):\n    def __init__(self, hid: int = 64, out: int = 64):\n        super().__init__()\n        # ① 1-hop\n        self.conv1 = HeteroConv({\n            ('user',   'ordered',      'order'):   SAGEConv((-1, -1), hid),\n            ('order',  'contains',     'product'): SAGEConv((-1, -1), hid),\n            ('order',  'ordered_by',   'user'):    SAGEConv((-1, -1), hid),  # 역\n            ('product','contained_in', 'order'):   SAGEConv((-1, -1), hid)   # 역\n        }, aggr='mean')\n        # ② 2-hop\n        self.conv2 = HeteroConv({\n            ('user',   'ordered',      'order'):   SAGEConv((hid, hid), out),\n            ('order',  'contains',     'product'): SAGEConv((hid, hid), out),\n            ('order',  'ordered_by',   'user'):    SAGEConv((hid, hid), out),\n            ('product','contained_in', 'order'):   SAGEConv((hid, hid), out)\n        }, aggr='mean')\n\n    def forward(self, x_dict, edge_dict):\n        h = {k: torch.relu(v) for 

In [11]:
data = HeteroData()
data['user'].x    = nn.Embedding(len(m2i), 32).weight
data['order'].x   = order_feat
data['product'].x = prod_feat
data['user'].num_nodes, data['order'].num_nodes, data['product'].num_nodes = \
    len(m2i), len(o2i), len(p2i)

# 구조적 간선
data['user','ordered','order'].edge_index        = uo
data['order','ordered_by','user'].edge_index     = uo.flip(0)
data['order','contains','product'].edge_index    = op
data['product','contained_in','order'].edge_index= op.flip(0)

# 레이블용 직접 user→product 간선
data['user','buys','product'].edge_index = edge_label_index

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = HetSAGE().to(device)

loader = LinkNeighborLoader(
    data,
    edge_label_index=(('user','buys','product'), edge_label_index),
    edge_label=edge_label,
    batch_size=1024,
    num_neighbors=[15,5],
    shuffle=True
)

# 학습 전에 더 이상 사용하지 않는 메모리 해제

In [12]:
def free_mem(*names):
    g = globals()
    for n in names:
        if n in g:
            del g[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [13]:
free_mem(
    'df_member', 'df_order', 'df_product',           # 노드 id DataFrame
    'df_order_feat', 'df_prod_vec',                  # 피처 생성용 DataFrame
    'df_edge_tr', 'df_label_tr',                     # 학습용 엣지/레이블 생성용
    'df_edge_te', 'df_label_te', 'df_gt',            # 테스트용 엣지/레이블, gt용
    'pos_u', 'pos_p', 'edge_pos', 'edge_neg',        # label/neg 임시 텐서
    'mid_arr', 'pid_arr'                             # ground-truth 변환 임시 배열
)

# 3. 학습

In [14]:
# ────────────────────────────────────────────────
# 하이퍼파라미터 및 환경 설정
# ────────────────────────────────────────────────
EPOCHS      = 100
CLIP_NORM   = 1.0
NEG_K       = 10
PATIENCE    = 20
TOP_K       = 5
INIT_LR     = 1e-3
BATCH_SIZE  = 1024       # 학습 미니배치 크기
VAL_BATCH   = 10000       # 검증 미니배치 크기
CKPT_DIR    = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = HetSAGE().to(device)

opt = torch.optim.Adam(model.parameters(), lr=INIT_LR, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
scaler = GradScaler()

BEST_PREC   = 0.0
no_improve  = 0

# (1) 전체 인덱스 → 실제 ID 역매핑 함수(딕셔너리)
m2i_inv = {v: k for k, v in m2i.items()}
o2i_inv = {v: k for k, v in o2i.items()}
p2i_inv = {v: k for k, v in p2i.items()}

# ────────────────────────────────────────────────
# 1. 부분 그래프 생성 함수 (1000명 단위로 미니배치 검증)
# ────────────────────────────────────────────────
def build_subgraph_for_users(user_ids, uo_te, op_te, order_feat, prod_feat, m2i, o2i, p2i, gt_dict):
    # 1) 포함 user 인덱스 추출
    user_idx = np.array([m2i[u] for u in user_ids])
    # 2) user-order 간선 필터
    mask_user = np.isin(uo_te[0].cpu().numpy(), user_idx)
    uo_sub = uo_te[:, mask_user]
    order_idx = np.unique(uo_sub[1].cpu().numpy())
    # 3) order-product 간선 필터
    mask_order = np.isin(op_te[0].cpu().numpy(), order_idx)
    op_sub = op_te[:, mask_order]
    product_idx = np.unique(op_sub[1].cpu().numpy())

    # 4) 새 인덱스 부여 (부분 그래프 기준 0~N)
    old2new_user   = {old: new for new, old in enumerate(user_idx)}
    old2new_order  = {old: new for new, old in enumerate(order_idx)}
    old2new_prod   = {old: new for new, old in enumerate(product_idx)}

    # 5) 리매핑 edge_index
    uo_sub_remap = torch.stack([
        torch.tensor([old2new_user[u.item()] for u in uo_sub[0]]),
        torch.tensor([old2new_order[o.item()] for o in uo_sub[1]])
    ], 0)
    op_sub_remap = torch.stack([
        torch.tensor([old2new_order[o.item()] for o in op_sub[0]]),
        torch.tensor([old2new_prod[p.item()] for p in op_sub[1]])
    ], 0)

    # 6) feature 추출
    order_x   = torch.tensor(np.stack([order_feat[o] for o in order_idx]), dtype=torch.float32)
    product_x = torch.tensor(np.stack([prod_feat[p] for p in product_idx]), dtype=torch.float32)

    # 7) gt_dict (부분 그래프 인덱스로 변환)
    gt_dict_remap = {
        old2new_user[m2i[u]]: [old2new_prod[p] for p in gt_dict[u] if p in old2new_prod]
        for u in user_ids if m2i[u] in old2new_user
    }

    # 8) HeteroData 객체 생성
    data = HeteroData()
    data['user'].num_nodes    = len(user_idx)
    data['order'].num_nodes   = len(order_idx)
    data['product'].num_nodes = len(product_idx)
    data['user'].x    = torch.nn.Embedding(len(user_idx), 32).weight
    data['order'].x   = order_x
    data['product'].x = product_x
    data['user','ordered','order'].edge_index         = uo_sub_remap
    data['order','ordered_by','user'].edge_index      = uo_sub_remap.flip(0)
    data['order','contains','product'].edge_index     = op_sub_remap
    data['product','contained_in','order'].edge_index = op_sub_remap.flip(0)
    return data, gt_dict_remap

# ────────────────────────────────────────────────
# 2. 미니배치 검증 함수
# ────────────────────────────────────────────────
@torch.no_grad()
def validate_minibatch(model, user_ids, uo_te, op_te, order_feat, prod_feat, m2i, o2i, p2i, gt_dict, TOP_K=5):
    """
    user_ids: 전역 user_id(원본 인덱스) 리스트 중 1000개씩 샘플
    gt_dict: 전역 user_id 기준
    나머지: 전체 그래프 및 피처
    """
    # 부분 그래프 생성
    batch_data, gt_dict_remap = build_subgraph_for_users(
        user_ids, uo_te, op_te, order_feat, prod_feat, m2i, o2i, p2i, gt_dict
    )
    batch_data = batch_data.to(device)
    model.eval()
    z = model(batch_data.x_dict, batch_data.edge_index_dict)
    uemb = z['user'].cpu()
    pemb = z['product'].cpu()
    CHUNK = 200_000

    precs = []
    for uid in range(len(user_ids)):
        if uid not in gt_dict_remap: continue
        gt = gt_dict_remap[uid]
        u = uemb[uid]
        scores_all, idx_all = [], []
        for s in range(0, len(pemb), CHUNK):
            sc, ix = (pemb[s:s+CHUNK] @ u).topk(TOP_K)
            scores_all.append(sc); idx_all.append(ix + s)
        scores_all = torch.cat(scores_all)
        idx_all = torch.cat(idx_all)
        topk_idx = scores_all.topk(TOP_K).indices
        top_pids = idx_all[topk_idx].numpy()
        hit = len(set(top_pids) & set(gt))
        precs.append(hit / TOP_K)
    return float(np.mean(precs))

# ────────────────────────────────────────────────
# 3. 학습 루프 (매 epoch마다 10000명 미니배치 검증)
# ────────────────────────────────────────────────

test_user_ids = list(gt_dict.keys())  # 75000명 유저
for ep in range(1, EPOCHS + 1):
    t_ep = time.perf_counter()
    running = 0.0
    total_samples = 0
    model.train()

    pbar = tqdm.tqdm(loader, desc=f"[Train] Epoch {ep}", ncols=100)
    for batch in pbar:
        batch = batch.to(device, non_blocking=True)
        bs = batch['user', 'buys', 'product'].edge_label.size(0)
        with torch.amp.autocast(device_type='cuda'):
            z = model(batch.x_dict, batch.edge_index_dict)
            src = z['user'][batch['user', 'buys', 'product'].edge_label_index[0]]
            dst = z['product'][batch['user', 'buys', 'product'].edge_label_index[1]]
            logits = (src * dst).sum(-1)
            loss = F.binary_cross_entropy_with_logits(
                logits,
                batch['user', 'buys', 'product'].edge_label,
                pos_weight=torch.tensor(NEG_K, device=device)
            )
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

        running += loss.item() * bs
        total_samples += bs
        if pbar.n:
            pbar.set_postfix(train_loss=f"{running/total_samples:.4f}")
        torch.cuda.empty_cache()

    train_loss = running / total_samples
    train_time = time.perf_counter() - t_ep
    print(f"→ Epoch {ep} done: loss={train_loss:.4f} | {train_time:.1f}s")

    # ────── 미니배치 검증(10000명 랜덤 샘플) ──────
    np.random.seed(ep)
    sample_users = np.random.choice(test_user_ids, VAL_BATCH, replace=False)
    val_prec = validate_minibatch(
        model, sample_users, uo_te, op_te, order_feat, prod_feat,
        m2i, o2i, p2i, gt_dict, TOP_K=TOP_K
    )
    print(f"→ Epoch {ep} validation Precision@{TOP_K} (minibatch) = {val_prec:.4f}")

    scheduler.step(val_prec)
    if val_prec > BEST_PREC:
        BEST_PREC = val_prec
        no_improve = 0
        ckpt_path = os.path.join(CKPT_DIR, "best_model_v0.1.pt") # save best model only
        torch.save({
            "epoch": ep,
            "model_state": model.state_dict(),
            "optimizer": opt.state_dict(),
            "scaler": scaler.state_dict(),
            "m2i": m2i, "o2i": o2i, "p2i": p2i,
            "precision": BEST_PREC,
        }, ckpt_path)
        print(f"✔  New best model saved: {ckpt_path}")
    else:
        no_improve += 1
        print(f"✘  No improvement for {no_improve}/{PATIENCE} epochs")

    if no_improve >= PATIENCE:
        print(f"🛑  Early stopping at epoch {ep}")
        break


[Train] Epoch 1: 100%|███████████████████████| 8275/8275 [11:04<00:00, 12.45it/s, train_loss=1.1748]


→ Epoch 1 done: loss=1.1748 | 664.6s
→ Epoch 1 validation Precision@5 (minibatch) = 0.0017
✔  New best model saved: checkpoints/best_model.pt


[Train] Epoch 2: 100%|███████████████████████| 8275/8275 [11:04<00:00, 12.45it/s, train_loss=1.1152]


→ Epoch 2 done: loss=1.1152 | 664.8s
→ Epoch 2 validation Precision@5 (minibatch) = 0.0057
✔  New best model saved: checkpoints/best_model.pt


[Train] Epoch 3: 100%|███████████████████████| 8275/8275 [11:04<00:00, 12.45it/s, train_loss=1.1158]


...

→ Epoch 98 done: loss=1.0436 | 671.1s
→ Epoch 98 validation Precision@5 (minibatch) = 0.0868
✘  No improvement for 8/20 epochs


[Train] Epoch 99: 100%|██████████████████████| 8275/8275 [10:51<00:00, 12.69it/s, train_loss=1.0436]


→ Epoch 99 done: loss=1.0436 | 652.0s
→ Epoch 99 validation Precision@5 (minibatch) = 0.0898
✘  No improvement for 9/20 epochs


[Train] Epoch 100: 100%|█████████████████████| 8275/8275 [10:52<00:00, 12.69it/s, train_loss=1.0441]


→ Epoch 100 done: loss=1.0441 | 652.3s
→ Epoch 100 validation Precision@5 (minibatch) = 0.0807
✘  No improvement for 10/20 epochs


### 현재 메모리에 남아 있는 모델 저장

In [15]:
"""
save_dir  = "checkpoints"
os.makedirs(save_dir, exist_ok=True)
stamp     = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
ckpt_path = os.path.join(save_dir, f"HetSAGE_ep{EPOCHS}_{stamp}.pt")

# 2) state_dict 패키징  ─────────────────────────────────────
torch.save({
    "epoch"       : EPOCHS,
    "model_state" : model.state_dict(),     # GNN 가중치
    "optimizer"   : opt.state_dict(),       # 옵티마이저 내부 momentum 등
    "scaler"      : scaler.state_dict(),    # AMP GradScaler 상태
    "m2i"         : m2i,
    "o2i"         : o2i,
    "p2i"         : p2i,
    "hid_dim"     : 64,                     # 하이퍼파라미터 메모
    "out_dim"     : 64,
    "neg_k"       : NEG_K,
    "num_layers"  : 2,
}, ckpt_path)

print(f"📦  모델 체크포인트 저장 완료 →  {ckpt_path}")\
"""

'\nsave_dir  = "checkpoints"\nos.makedirs(save_dir, exist_ok=True)\nstamp     = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")\nckpt_path = os.path.join(save_dir, f"HetSAGE_ep{EPOCHS}_{stamp}.pt")\n\n# 2) state_dict 패키징  ─────────────────────────────────────\ntorch.save({\n    "epoch"       : EPOCHS,\n    "model_state" : model.state_dict(),     # GNN 가중치\n    "optimizer"   : opt.state_dict(),       # 옵티마이저 내부 momentum 등\n    "scaler"      : scaler.state_dict(),    # AMP GradScaler 상태\n    "m2i"         : m2i,\n    "o2i"         : o2i,\n    "p2i"         : p2i,\n    "hid_dim"     : 64,                     # 하이퍼파라미터 메모\n    "out_dim"     : 64,\n    "neg_k"       : NEG_K,\n    "num_layers"  : 2,\n}, ckpt_path)\n\nprint(f"📦  모델 체크포인트 저장 완료 →  {ckpt_path}")'

# 4. 평가

### 메모리 해제(학습 시 사용한 메모리)

In [16]:
free_mem(
    'data',                 # 학습용 HeteroData
    'loader',               # 학습용 LinkNeighborLoader
    'edge_label_index',     # 학습용 직접엣지 인덱스
    'edge_label',           # 학습용 라벨
    'batch',                # 마지막 배치 임시 텐서(있다면)
)
print("🗑️  학습 전용 객체 해제 완료 — 모델·피처는 유지 중")

🗑️  학습 전용 객체 해제 완료 — 모델·피처는 유지 중


In [17]:
@torch.no_grad()
def evaluate(model, data, gt_dict, k=TOP_K):
    model.eval()
    data = data.to(device)
    z    = model(data.x_dict, data.edge_index_dict)
    uemb = z['user'].cpu()
    pemb = z['product'].cpu()

    precs = []
    for mid, gt in gt_dict.items():
        if mid not in m2i: continue
        u = uemb[m2i[mid]]
        # chunk-wise TopK
        scores_all, idx_all = [], []
        for s in range(0, len(pemb), CHUNK):
            sc, ix = (pemb[s:s+CHUNK] @ u).topk(k)
            scores_all.append(sc); idx_all.append(ix + s)
        scores_all = torch.cat(scores_all)
        idx_all    = torch.cat(idx_all)
        topk_idx   = scores_all.topk(k).indices
        top_pids   = i2p[idx_all[topk_idx].numpy()]

        hit = len(set(top_pids) & set(gt))
        precs.append(hit / k)
    return float(np.mean(precs))

t0 = time.perf_counter()
mean_p5 = evaluate(model, test_data, gt_dict, TOP_K)
log(f"Precision@{TOP_K} = {mean_p5:.4f}", t0)

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.60 GiB. GPU 0 has a total capacity of 7.75 GiB of which 1.34 GiB is free. Process 555417 has 760.00 MiB memory in use. Including non-PyTorch memory, this process has 4.89 GiB memory in use. Of the allocated memory 4.38 GiB is allocated by PyTorch, and 315.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
"""
# test-prior 간선 로드  (eval_set=test & role_test=prior)
df_edge_te = cypher_df("""
#MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
#WHERE o.eval_set = 'test' AND o.role_test = 'prior'
#RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")

uo_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid"))
), 0).to(torch.long)

op_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid")),
    torch.from_numpy(safe_map(df_edge_te.pid, p2i, "pid"))
), 0).to(torch.long)

# 1. 테스트용 HeteroData  (정방향 + 역방향 간선 유지)
test_data = HeteroData()
test_data['user'].num_nodes, test_data['order'].num_nodes, test_data['product'].num_nodes = \
    len(m2i), len(o2i), len(p2i)

test_data['user'].x    = data['user'].x          # 학습 때 만든 노드 특성 재사용
test_data['order'].x   = data['order'].x
test_data['product'].x = data['product'].x

test_data['user','ordered','order'].edge_index         = uo_te
test_data['order','ordered_by','user'].edge_index      = uo_te.flip(0)
test_data['order','contains','product'].edge_index     = op_te
test_data['product','contained_in','order'].edge_index = op_te.flip(0)

# Precision@K 평가 
t0 = time.perf_counter()
mean_p5 = precision_at_k(model, test_data, gt_dict, TOP_K)
log(f"Precision@{TOP_K} = {mean_p5:.4f}", t0)
"""